<a href="https://colab.research.google.com/github/abhikumar63/ML_Practice/blob/main/MovingAveragecuDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import cudf
import numpy as np
import numba.cuda
import time

In [18]:
data_length = int(1e8)
average_window = 4
threads_per_block = 128
chunk_size = 10240

In [29]:
df = cudf.DataFrame()
df['in1'] = np.arange(data_length, dtype=np.float64)

In [30]:
# Ensure GPU is synchronized before measuring
numba.cuda.synchronize()
start = time.time()

# Standard, modern and optimized cuDF rolling moving average
df['out'] = df['in1'].rolling(window=average_window).mean()

# Synchronize GPU to ensure calculation finished
numba.cuda.synchronize()
end = time.time()

print('cuDF rolling time:', end - start)
display(df.head(10))
display(df.tail(10))

cuDF rolling time: 0.09644389152526855


,in1,out
0,0.0,<NA>
1,1.0,<NA>
2,2.0,<NA>
3,3.0,1.5
4,4.0,2.5
5,5.0,3.5
6,6.0,4.5
7,7.0,5.5
8,8.0,6.5
9,9.0,7.5


,in1,out
99999990,99999990.0,99999988.5
99999991,99999991.0,99999989.5
99999992,99999992.0,99999990.5
99999993,99999993.0,99999991.5
99999994,99999994.0,99999992.5
99999995,99999995.0,99999993.5
99999996,99999996.0,99999994.5
99999997,99999997.0,99999995.5
99999998,99999998.0,99999996.5
99999999,99999999.0,99999997.5


In [31]:
import numba.cuda as cuda
import math

@cuda.jit
def custom_moving_average_kernel(in1, out, window_size):
    # Calculate the global thread index
    idx = cuda.grid(1)

    # Boundary guard
    if idx < in1.size:
        if idx < window_size - 1:
            # Not enough elements for a full window
            out[idx] = np.nan
        else:
            # Calculate the sum for the window
            summ = 0.0
            for i in range(idx - window_size + 1, idx + 1):
                summ += in1[i]

            # Apply the custom formula: avg + 10
            out[idx] = (summ / window_size) + 10.0


In [32]:
# Synchronize GPU before measuring
numba.cuda.synchronize()
start = time.time()

# 1. Allocate space on the GPU for the output array
# Convert cuDF Series directly to Numba device arrays without copying data
d_in = cuda.as_cuda_array(df['in1'].to_cupy())
d_out = cuda.device_array_like(d_in)

# 2. Configure the CUDA Grid structure
threads_per_block = 256
blocks_per_grid = (data_length + (threads_per_block - 1)) // threads_per_block

# 3. Launch the compiled CUDA kernel
custom_moving_average_kernel[blocks_per_grid, threads_per_block](
    d_in, d_out, average_window
)

# 4. Copy the result back to our DataFrame
df['custom_out'] = cudf.Series(d_out)

# Synchronize GPU to finish measuring
numba.cuda.synchronize()
end = time.time()

print('Custom CUDA JIT Kernel Time:', end - start)
display(df.head(10))


Custom CUDA JIT Kernel Time: 1.7578582763671875


,in1,out,custom_out
0,0.0,<NA>,<NA>
1,1.0,<NA>,<NA>
2,2.0,<NA>,<NA>
3,3.0,1.5,11.5
4,4.0,2.5,12.5
5,5.0,3.5,13.5
6,6.0,4.5,14.5
7,7.0,5.5,15.5
8,8.0,6.5,16.5
9,9.0,7.5,17.5


### 1. Write the Native C++ CUDA Kernel
We will write the CUDA source code to a file named `custom_kernel.cu`. It takes raw device pointers directly.

In [33]:
%%writefile custom_kernel.cu
#include <cuda_runtime.h>
#include <cmath>

__global__
void moving_average_cpp_kernel(const double* in1, double* out, int data_length, int window_size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < data_length) {
        if (idx < window_size - 1) {
            out[idx] = NAN;
        } else {
            double sum = 0.0;
            for (int i = idx - window_size + 1; i <= idx; ++i) {
                sum += in1[i];
            }
            // Apply custom formula: avg + 10
            out[idx] = (sum / window_size) + 10.0;
        }
    }
}

// Extern "C" wrapper to prevent name mangling, allowing ctypes to locate the function
extern "C" {
    void launch_custom_kernel(const double* in1, double* out, int data_length, int window_size, int threads_per_block) {
        int blocks_per_grid = (data_length + threads_per_block - 1) / threads_per_block;
        moving_average_cpp_kernel<<<blocks_per_grid, threads_per_block>>>(in1, out, data_length, window_size);
        cudaDeviceSynchronize();
    }
}

Writing custom_kernel.cu


### 2. Compile C++ Code to a Shared Library (`.so`)

In [34]:
!nvcc -O3 -shared -Xcompiler -fPIC -o libcustom_kernel.so custom_kernel.cu

### 3. Load the C++ Library and Run it on cuDF Device Pointers

In [36]:
import ctypes

# Load the compiled shared library
libcudakernel = ctypes.CDLL('./libcustom_kernel.so')

# Define the argument types for our wrapper function
libcudakernel.launch_custom_kernel.argtypes = [
    ctypes.c_void_p, # double* in1 (device pointer)
    ctypes.c_void_p, # double* out (device pointer)
    ctypes.c_int,    # data_length
    ctypes.c_int,    # window_size
    ctypes.c_int     # threads_per_block
]

# Allocate output GPU memory in cuDF
df['cpp_custom_out'] = cudf.Series(np.empty(data_length, dtype=np.float64))

# Extract raw GPU memory addresses (device pointers) from cuDF
in_ptr = df['in1'].__cuda_array_interface__['data'][0]
out_ptr = df['cpp_custom_out'].__cuda_array_interface__['data'][0]

# Synchronize and time the execution of native C++ code
numba.cuda.synchronize()
start = time.time()

libcudakernel.launch_custom_kernel(
    in_ptr,
    out_ptr,
    data_length,
    average_window,
    256 # threads_per_block
)

numba.cuda.synchronize()
end = time.time()

print('Native C++ CUDA Kernel Time:', end - start)
display(df.head(10))
display(df.tail(10))

Native C++ CUDA Kernel Time: 0.03354525566101074


,in1,out,custom_out,cpp_custom_out
0,0.0,<NA>,<NA>,NaN
1,1.0,<NA>,<NA>,NaN
2,2.0,<NA>,<NA>,NaN
3,3.0,1.5,11.5,11.5
4,4.0,2.5,12.5,12.5
5,5.0,3.5,13.5,13.5
6,6.0,4.5,14.5,14.5
7,7.0,5.5,15.5,15.5
8,8.0,6.5,16.5,16.5
9,9.0,7.5,17.5,17.5


,in1,out,custom_out,cpp_custom_out
99999990,99999990.0,99999988.5,99999998.5,99999998.5
99999991,99999991.0,99999989.5,99999999.5,99999999.5
99999992,99999992.0,99999990.5,100000000.5,100000000.5
99999993,99999993.0,99999991.5,100000001.5,100000001.5
99999994,99999994.0,99999992.5,100000002.5,100000002.5
99999995,99999995.0,99999993.5,100000003.5,100000003.5
99999996,99999996.0,99999994.5,100000004.5,100000004.5
99999997,99999997.0,99999995.5,100000005.5,100000005.5
99999998,99999998.0,99999996.5,100000006.5,100000006.5
99999999,99999999.0,99999997.5,100000007.5,100000007.5
